In [ ]:
import pandas as pd
import numpy as np
import os



In [ ]:
file_path = r'D:\temp\slabshicao\data\Report_P400003.csv'
df = pd.read_csv(file_path, sep=None, engine='python')
# 提取第二列 (U3)，转为数值
target_column = df.iloc[:, 1]
clean_data = pd.to_numeric(target_column, errors='coerce').dropna().values
# 3. 提取 41x41 的数据
data_matrix = []
for i in range(41):
    start_idx = i * 164
    end_idx = start_idx + 41
    row_values = clean_data[start_idx : end_idx]
    
    if len(row_values) == 41:
        data_matrix.append(row_values)
X_train = np.array(data_matrix)

In [ ]:
global_max = X_train.max()
global_min = X_train.min()
X_train = (X_train - global_min) /(global_max - global_min)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
plt.figure(figsize=(8, 7)) # 设置图的大小，你可以调整
heatmap = plt.imshow(X_train, cmap='jet', origin='lower')
# 添加颜色条
cbar = plt.colorbar(heatmap, shrink=0.8, aspect=20)
cbar.set_label('U3 Displacement (mm/m etc.)', fontsize=12) # 颜色条标签
# 设置图标题
plt.title('Surface U3 Displacement Heatmap', fontsize=14)
# 设置轴标签
plt.xlabel('X-direction Node Index', fontsize=12)
plt.ylabel('Y-direction Node Index', fontsize=1)

In [ ]:
import torch
from Network import VAE_41

# 加载模型权重
model = VAE_41()   # 这里应该是你的模型定义
model.load_state_dict(torch.load('./weight/size41_0119test.pth'))  # 加载预训练的权重
model.eval()  # 切换到评估模式


In [ ]:
data_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(0).unsqueeze(0) 
print(data_tensor.shape)


In [ ]:
# 通过VAE进行推理，获取重建结果
with torch.no_grad():  # 在推理时不计算梯度
    reconstructed_data, _, _ = model(data_tensor)  # 假设VAE模型返回重建数据

In [ ]:
import matplotlib.pyplot as plt

# 将重建结果转换回 NumPy 数组并显示
reconstructed_data_np = reconstructed_data.squeeze(0).squeeze(0).cpu().numpy()  # 去掉 batch 和 channel 维度

# 可视化原始数据和重建数据
plt.figure(figsize=(10, 5))

# 原始数据
plt.subplot(1, 2, 1)
plt.imshow(X_train, vmin=0, vmax=1, cmap='viridis')
plt.title('Original Data')
plt.colorbar()

# 重建数据
plt.subplot(1, 2, 2)
plt.imshow(reconstructed_data_np, vmin=0, vmax=1, cmap='viridis')
plt.title('Reconstructed Data')
plt.colorbar()

plt.show()
